# TFiltersPy Benchmark — All Filters Compared

This notebook benchmarks all five Bayesian state-estimation filters provided by TFiltersPy
on an **image denoising** task using the scikit-learn digits dataset (8x8 pixel images, 64 dimensions).

Filters under test:

| Filter | Abbreviation | Key property |
|---|---|---|
| KalmanFilter | KF | Linear, closed-form |
| ExtendedKalmanFilter | EKF | Nonlinear, Jacobian-based |
| UnscentedKalmanFilter | UKF | Nonlinear, sigma-point |
| EnsembleKalmanFilter | EnKF | Monte Carlo ensemble |
| ParticleFilter | PF | Sequential Monte Carlo |

In [ ]:
import time
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt

from tfilterspy import (
    KalmanFilter,
    ExtendedKalmanFilter,
    UnscentedKalmanFilter,
    EnsembleKalmanFilter,
    ParticleFilter,
)

In [ ]:
# --- Data preparation ---
digits = load_digits()
X_all = digits.data[:200]      # 200 samples, 64 features each
y_all = digits.target[:200]

X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

# Add Gaussian noise
rng = np.random.RandomState(0)
noise_level = 0.88
noisy_train = X_train + rng.normal(0, noise_level, X_train.shape)
noisy_test  = X_test  + rng.normal(0, noise_level, X_test.shape)

# --- Common filter parameters ---
n_features = X_train.shape[1]  # 64
F  = np.eye(n_features) * 0.99
H  = np.eye(n_features)
Q  = np.eye(n_features) * 0.01
R  = np.eye(n_features) * 0.1
x0 = X_train[0].copy()
P0 = np.eye(n_features)

print(f"Training samples : {len(X_train)}")
print(f"Test samples     : {len(X_test)}")
print(f"Feature dimension: {n_features}")
print(f"Noise std dev    : {noise_level}")

In [ ]:
def benchmark_filter(name, filter_obj, noisy_data, clean_data, n_runs=3):
    """Run a filter multiple times and collect timing / MSE statistics."""
    times = []
    mses  = []
    for _ in range(n_runs):
        start = time.time()
        filter_obj.fit(noisy_data)
        denoised = filter_obj.predict()
        elapsed = time.time() - start
        times.append(elapsed)
        mses.append(np.mean((clean_data - denoised) ** 2))
    return {
        "name":      name,
        "mean_time": np.mean(times),
        "std_time":  np.std(times),
        "mean_mse":  np.mean(mses),
        "std_mse":   np.std(mses),
    }

In [ ]:
# --- Build all five filters ---
filters = {
    "KF": KalmanFilter(F, H, Q, R, x0, P0),

    "EKF": ExtendedKalmanFilter(
        f=lambda x: F @ x,
        h=lambda x: H @ x,
        F_jacobian=lambda x: F,
        H_jacobian=lambda x: H,
        Q=Q, R=R, x0=x0, P0=P0,
    ),

    "UKF": UnscentedKalmanFilter(
        f=lambda x: F @ x,
        h=lambda x: H @ x,
        Q=Q, R=R, x0=x0, P0=P0,
    ),

    "EnKF": EnsembleKalmanFilter(
        f=lambda x: F @ x,
        h=lambda x: H @ x,
        Q=Q, R=R, x0=x0, P0=P0,
        n_ensemble=50, use_dask=False,
    ),

    "PF": ParticleFilter(
        f=F, h=H, Q=Q, R=R, x0=x0,
        n_particles=200,
    ),
}

# --- Run benchmarks ---
results = []
for name, filt in filters.items():
    print(f"Benchmarking {name} ...")
    res = benchmark_filter(name, filt, noisy_train, X_train, n_runs=3)
    results.append(res)
    print(f"  Time: {res['mean_time']:.4f}s (+/- {res['std_time']:.4f})  "
          f"MSE: {res['mean_mse']:.4f} (+/- {res['std_mse']:.4f})")

# --- Print summary table ---
print("\n" + "-" * 62)
print(f"{'Filter':<8} {'Time (s)':>12} {'Std':>10} {'MSE':>12} {'Std':>10}")
print("-" * 62)
for r in results:
    print(f"{r['name']:<8} {r['mean_time']:>12.4f} {r['std_time']:>10.4f} "
          f"{r['mean_mse']:>12.4f} {r['std_mse']:>10.4f}")
print("-" * 62)

In [ ]:
# --- Visualization: side-by-side bar charts ---
names      = [r["name"] for r in results]
mean_times = [r["mean_time"] for r in results]
std_times  = [r["std_time"]  for r in results]
mean_mses  = [r["mean_mse"]  for r in results]
std_mses   = [r["std_mse"]   for r in results]

colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd"]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Execution time ---
bars1 = ax1.bar(names, mean_times, yerr=std_times, capsize=5,
                color=colors, alpha=0.85)
ax1.set_xlabel("Filter")
ax1.set_ylabel("Execution Time (seconds)")
ax1.set_title("Execution Time by Filter")
ax1.grid(axis="y", linestyle="--", alpha=0.6)
for bar, val in zip(bars1, mean_times):
    ax1.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f"{val:.3f}s", ha="center", va="bottom", fontsize=9)

# --- MSE ---
bars2 = ax2.bar(names, mean_mses, yerr=std_mses, capsize=5,
                color=colors, alpha=0.85)
ax2.set_xlabel("Filter")
ax2.set_ylabel("Mean Squared Error")
ax2.set_title("Denoising MSE by Filter")
ax2.grid(axis="y", linestyle="--", alpha=0.6)
for bar, val in zip(bars2, mean_mses):
    ax2.text(bar.get_x() + bar.get_width() / 2, bar.get_height(),
             f"{val:.4f}", ha="center", va="bottom", fontsize=9)

fig.suptitle("TFiltersPy Benchmark: Digits Image Denoising", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# --- Classification accuracy on denoised data ---
# Baseline: accuracy on noisy data (no filtering)
clf_noisy = LogisticRegression(max_iter=5000, random_state=42)
clf_noisy.fit(noisy_train, y_train)
acc_noisy = clf_noisy.score(noisy_test, y_test)
print(f"{'Noisy (no filter)':<20s}  Accuracy: {acc_noisy:.4f}")

# Baseline: accuracy on clean data
clf_clean = LogisticRegression(max_iter=5000, random_state=42)
clf_clean.fit(X_train, y_train)
acc_clean = clf_clean.score(X_test, y_test)
print(f"{'Clean (oracle)':<20s}  Accuracy: {acc_clean:.4f}")

print("-" * 42)

accuracies = {}
for name, filt in filters.items():
    # Denoise training data
    filt.fit(noisy_train)
    denoised_train = filt.predict()

    # Denoise test data
    filt.fit(noisy_test)
    denoised_test = filt.predict()

    clf = LogisticRegression(max_iter=5000, random_state=42)
    clf.fit(denoised_train, y_train)
    acc = clf.score(denoised_test, y_test)
    accuracies[name] = acc
    print(f"{name:<20s}  Accuracy: {acc:.4f}")

## Summary: When to Use Each Filter

| Filter | Speed | MSE | Best suited for |
|--------|-------|-----|------------------|
| **KalmanFilter** | Fastest | Optimal (linear) | Linear systems with Gaussian noise; real-time applications |
| **ExtendedKalmanFilter** | Fast | Near-optimal | Mildly nonlinear systems where Jacobians are available |
| **UnscentedKalmanFilter** | Moderate | Near-optimal | Nonlinear systems; no Jacobians needed |
| **EnsembleKalmanFilter** | Moderate | Good | High-dimensional nonlinear systems; parallelizable via Dask |
| **ParticleFilter** | Slowest | Varies | Non-Gaussian / highly nonlinear systems; multi-modal posteriors |

**Key takeaways:**

- For linear or nearly-linear problems, `KalmanFilter` is hard to beat in both speed and accuracy.
- `EKF` and `UKF` offer good nonlinear performance; prefer `UKF` when Jacobians are hard to derive.
- `EnKF` scales to very high dimensions where storing full covariance matrices is impractical.
- `ParticleFilter` is the most flexible but slowest; use when the posterior is non-Gaussian or multi-modal.